# UQ-SHRED Experiments: Solar Data (SUN)

Same experimental suite as the FLOW notebook, adapted for solar imagery:
- **E1:** Reconstruction (SHRED baseline vs UQ-SHRED with uncertainty)
- **E2:** Calibration
- **E3:** Uncertainty-Error Correlation
- **E4:** Spatial Uncertainty (2D image maps)
- **E5:** Forecasting with uncertainty
- **E6:** Ablation (sampling size)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import MinMaxScaler
import os
from datetime import datetime
import json

from processdata import load_data, TimeSeriesDataset
from models import SHRED, UQ_SHRED, UQ_Forecaster, fit, fit_uq
import uq

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset_name = 'SUN'
IMG_H, IMG_W = 271, 271  # solar image spatial dimensions
print(f'Using device: {device}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'results/{dataset_name}_{timestamp}'
os.makedirs(results_dir, exist_ok=True)
print(f'Results: {results_dir}')

np.random.seed(42)
torch.manual_seed(42)

config = {'dataset': dataset_name, 'device': device, 'seed': 42, 'timestamp': timestamp}

In [ ]:
# Data
num_sensors = 3
lags = 75

load_X = load_data(dataset_name)  # shape: (N, 271*271)
n, m = load_X.shape
print(f'{dataset_name} shape: {load_X.shape}  (spatial: {IMG_H}x{IMG_W}={m})')

sensor_locations = np.random.choice(m, size=num_sensors, replace=False)
config.update({'num_sensors': num_sensors, 'lags': lags, 'data_shape': list(load_X.shape)})

In [ ]:
# Train/valid/test split
train_indices = np.random.choice(n - lags, size=int(0.7*(n - lags)), replace=False)
mask = np.ones(n - lags)
mask[train_indices] = 0
valid_test_indices = np.arange(0, n - lags)[np.where(mask != 0)[0]]
valid_indices = valid_test_indices[::2]
test_indices = valid_test_indices[1::2]

print(f'Train: {len(train_indices)}, Valid: {len(valid_indices)}, Test: {len(test_indices)}')

config['split'] = {'train': len(train_indices), 'valid': len(valid_indices), 'test': len(test_indices)}

In [ ]:
# Normalize
sc = MinMaxScaler()
sc = sc.fit(load_X[train_indices])
transformed_X = sc.transform(load_X)

all_data_in = np.zeros((n - lags, lags, num_sensors))
for i in range(len(all_data_in)):
    all_data_in[i] = transformed_X[i:i+lags, sensor_locations]

train_data_in = torch.tensor(all_data_in[train_indices], dtype=torch.float32).to(device)
valid_data_in = torch.tensor(all_data_in[valid_indices], dtype=torch.float32).to(device)
test_data_in  = torch.tensor(all_data_in[test_indices],  dtype=torch.float32).to(device)

train_data_out = torch.tensor(transformed_X[train_indices + lags - 1], dtype=torch.float32).to(device)
valid_data_out = torch.tensor(transformed_X[valid_indices + lags - 1], dtype=torch.float32).to(device)
test_data_out  = torch.tensor(transformed_X[test_indices  + lags - 1], dtype=torch.float32).to(device)

train_dataset = TimeSeriesDataset(train_data_in, train_data_out)
valid_dataset = TimeSeriesDataset(valid_data_in, valid_data_out)
test_dataset  = TimeSeriesDataset(test_data_in,  test_data_out)

print(f'Input: {train_data_in.shape}, Output: {train_data_out.shape}')

# E1: Reconstruction

In [ ]:
# SHRED baseline
shred = SHRED(num_sensors, m, hidden_size=128, hidden_layers=2, l1=350, l2=400, dropout=0.01).to(device)
fit(shred, train_dataset, valid_dataset, batch_size=20, num_epochs=1000, lr=1e-3, verbose=True, patience=50)

shred.eval()
with torch.no_grad():
    shred_recon = shred(test_dataset.X)
    shred_error = (torch.linalg.norm(shred_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()
print(f'SHRED relative error: {shred_error:.4f}')

In [ ]:
# UQ-SHRED
uq_shred = UQ_SHRED(num_sensors, m, hidden_size=128, hidden_layers=2, l1=350, l2=400, dropout=0.1, noise_dim=100).to(device)
fit_uq(uq_shred, train_dataset, valid_dataset, batch_size=20, num_epochs=1000, lr=1e-3, verbose=True, patience=50)

uq_shred.eval()
samples = uq_shred.sample(test_dataset.X, n_samples=50)
samples_np = samples.cpu().numpy()

mean_recon   = samples.mean(dim=0)
median_recon = torch.tensor(np.median(samples_np, axis=0), dtype=torch.float32).to(device)
std_recon    = samples.std(dim=0)

uq_mean_error   = (torch.linalg.norm(mean_recon   - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()
uq_median_error = (torch.linalg.norm(median_recon - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()

print(f'UQ Mean: {uq_mean_error:.4f}, UQ Median: {uq_median_error:.4f}')

In [ ]:
# Metrics
crps_score = uq.crps(samples, test_dataset.Y)
sharp      = uq.sharpness(samples, conf=0.95)
cal_scores = uq.calibration_scores(samples, test_dataset.Y, levels=[0.5, 0.7, 0.9, 0.95, 0.99])

errors = torch.abs(test_dataset.Y - mean_recon).flatten().cpu().numpy()
stds   = std_recon.flatten().cpu().numpy()
corr   = np.corrcoef(stds, errors)[0, 1]

print('\n=== E1 Metrics ===')
print(f'SHRED:      {shred_error:.4f}')
print(f'UQ Mean:    {uq_mean_error:.4f}')
print(f'UQ Median:  {uq_median_error:.4f}')
print(f'CRPS:       {crps_score:.4f}')
print(f'Sharpness:  {sharp:.4f}')
print(f'Coverage:   {cal_scores[0.95]*100:.1f}%')
print(f'Corr:       {corr:.3f}')

In [ ]:
# E1 Visualization: 2D solar image reconstruction
shred_recon_np = sc.inverse_transform(shred_recon.cpu().numpy())
test_truth_np  = sc.inverse_transform(test_dataset.Y.cpu().numpy())
samples_orig   = np.array([sc.inverse_transform(s) for s in samples_np])
mean_orig      = samples_orig.mean(axis=0)
lower_orig     = np.percentile(samples_orig, 2.5,  axis=0)
upper_orig     = np.percentile(samples_orig, 97.5, axis=0)
std_orig       = samples_orig.std(axis=0)

# Pick snapshot t=0 from test set
t0 = 0
truth_img  = test_truth_np[t0].reshape(IMG_H, IMG_W)
shred_img  = shred_recon_np[t0].reshape(IMG_H, IMG_W)
mean_img   = mean_orig[t0].reshape(IMG_H, IMG_W)
std_img    = std_orig[t0].reshape(IMG_H, IMG_W)
ci_width   = (upper_orig[t0] - lower_orig[t0]).reshape(IMG_H, IMG_W)

vmin, vmax = truth_img.min(), truth_img.max()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

im0 = axes[0, 0].imshow(truth_img, cmap='plasma', vmin=vmin, vmax=vmax)
axes[0, 0].set_title('Ground Truth')
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(shred_img, cmap='plasma', vmin=vmin, vmax=vmax)
axes[0, 1].set_title('SHRED Reconstruction')
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(mean_img, cmap='plasma', vmin=vmin, vmax=vmax)
axes[0, 2].set_title('UQ-SHRED Mean')
plt.colorbar(im2, ax=axes[0, 2])

err_shred = np.abs(truth_img - shred_img)
err_uq    = np.abs(truth_img - mean_img)
emax = max(err_shred.max(), err_uq.max())

im3 = axes[1, 0].imshow(err_shred, cmap='hot', vmin=0, vmax=emax)
axes[1, 0].set_title('SHRED |Error|')
plt.colorbar(im3, ax=axes[1, 0])

im4 = axes[1, 1].imshow(err_uq, cmap='hot', vmin=0, vmax=emax)
axes[1, 1].set_title('UQ-SHRED |Error|')
plt.colorbar(im4, ax=axes[1, 1])

im5 = axes[1, 2].imshow(std_img, cmap='viridis')
axes[1, 2].set_title('UQ-SHRED Uncertainty (σ)')
plt.colorbar(im5, ax=axes[1, 2])

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('E1: Solar Image Reconstruction (t=0)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{results_dir}/E1_reconstruction_solar.png', dpi=150)
plt.show()

In [ ]:
# E1 Time-series view at a single pixel
pixel_idx = sensor_locations[1]
t_range = np.arange(50)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(t_range, test_truth_np[:50, pixel_idx], 'k-', label='Ground Truth')
axes[0].plot(t_range, shred_recon_np[:50, pixel_idx], 'r-', label='SHRED')
axes[0].set_ylabel('Intensity')
axes[0].legend()
axes[0].set_title('SHRED (Deterministic)')
axes[0].grid(alpha=0.3)

axes[1].fill_between(t_range, lower_orig[:50, pixel_idx], upper_orig[:50, pixel_idx],
                     alpha=0.3, color='blue', label='95% CI')
axes[1].plot(t_range, test_truth_np[:50, pixel_idx], 'k-', label='Ground Truth')
axes[1].plot(t_range, mean_orig[:50, pixel_idx],    'b-', label='UQ-SHRED Mean')
axes[1].set_ylabel('Intensity')
axes[1].legend()
axes[1].set_title('UQ-SHRED (With Uncertainty)')
axes[1].grid(alpha=0.3)

axes[2].plot(t_range, test_truth_np[:50, pixel_idx],   'k-',  linewidth=1.5, label='Ground Truth')
axes[2].plot(t_range, shred_recon_np[:50, pixel_idx],  'r-',  linewidth=1.2, label='SHRED')
axes[2].plot(t_range, mean_orig[:50, pixel_idx],        'b-',  linewidth=1.2, label='UQ Mean')
axes[2].fill_between(t_range, lower_orig[:50, pixel_idx], upper_orig[:50, pixel_idx],
                     alpha=0.15, color='blue')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Intensity')
axes[2].legend()
axes[2].set_title('E1: Overlay (sensor pixel)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/E1_timeseries_pixel.png', dpi=150)
plt.show()

# E2: Calibration

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
uq.plot_calibration(samples, test_dataset.Y, ax=ax)
ax.set_title('E2: Calibration (SUN)')
plt.savefig(f'{results_dir}/E2_calibration.png', dpi=150)
plt.show()

print('E2: Calibration')
for level, obs in cal_scores.items():
    print(f'  {level*100:.0f}% → {obs*100:.1f}%')

# E3: Uncertainty-Error Relationship

In [ ]:
idx = np.random.choice(len(errors), min(5000, len(errors)), replace=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(stds[idx], errors[idx], alpha=0.1, s=1)
z = np.polyfit(stds[idx], errors[idx], 1)
p = np.poly1d(z)
x_line = np.linspace(stds.min(), stds.max(), 100)
ax.plot(x_line, p(x_line), 'r-', linewidth=2)
ax.set_title(f'E3: Uncertainty vs Error (ρ={corr:.3f})')
ax.set_xlabel('Uncertainty (σ)')
ax.set_ylabel('Error')
ax.grid(alpha=0.3)
plt.savefig(f'{results_dir}/E3_uncertainty_vs_error.png', dpi=150)
plt.show()

print(f'E3: ρ = {corr:.4f}')

# E4: Spatial Uncertainty (2D Maps)

In [ ]:
avg_std_flat = std_recon.cpu().numpy().mean(axis=0)  # mean over test time steps
avg_std_img  = avg_std_flat.reshape(IMG_H, IMG_W)

avg_err_flat = errors.reshape(-1, m).mean(axis=0)  # mean abs error over test time
avg_err_img  = avg_err_flat.reshape(IMG_H, IMG_W)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Mean ground truth image (for spatial reference)
mean_truth_img = test_truth_np.mean(axis=0).reshape(IMG_H, IMG_W)
im0 = axes[0].imshow(mean_truth_img, cmap='plasma')
axes[0].set_title('Mean Solar Field (test set)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(avg_std_img, cmap='viridis')
axes[1].set_title('E4: Mean Uncertainty σ (spatial)')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(avg_err_img, cmap='hot')
axes[2].set_title('E4: Mean |Error| (spatial)')
plt.colorbar(im2, ax=axes[2])

for ax in axes:
    ax.axis('off')

plt.suptitle('E4: Spatial Uncertainty — Solar Data', fontsize=13)
plt.tight_layout()
plt.savefig(f'{results_dir}/E4_spatial_uncertainty_2d.png', dpi=150)
plt.show()

print(f'E4: Mean σ={avg_std_flat.mean():.4f}, Max σ={avg_std_flat.max():.4f}')

In [ ]:
# E4 Snapshot evolution: uncertainty at multiple time steps
t_snaps = [0, len(test_indices)//4, len(test_indices)//2, -1]
t_labels = ['t=0 (start)', 't=25%', 't=50%', 't=end']

fig, axes = plt.subplots(2, len(t_snaps), figsize=(16, 8))

all_stds_np = std_recon.cpu().numpy()

for col, (ti, label) in enumerate(zip(t_snaps, t_labels)):
    truth_snap = test_truth_np[ti].reshape(IMG_H, IMG_W)
    std_snap   = all_stds_np[ti].reshape(IMG_H, IMG_W)

    im_t = axes[0, col].imshow(truth_snap, cmap='plasma')
    axes[0, col].set_title(f'Truth {label}')
    axes[0, col].axis('off')
    plt.colorbar(im_t, ax=axes[0, col])

    im_s = axes[1, col].imshow(std_snap, cmap='viridis')
    axes[1, col].set_title(f'σ {label}')
    axes[1, col].axis('off')
    plt.colorbar(im_s, ax=axes[1, col])

plt.suptitle('E4: Solar Field & Uncertainty at Multiple Snapshots', fontsize=13)
plt.tight_layout()
plt.savefig(f'{results_dir}/E4_spatial_snapshots.png', dpi=150)
plt.show()

# E5: Forecasting

In [ ]:
sensor_data = transformed_X[:, sensor_locations]
forecast_in  = np.zeros((n - lags, lags, num_sensors))
forecast_out = np.zeros((n - lags, num_sensors))
for i in range(n - lags):
    forecast_in[i]  = sensor_data[i:i+lags]
    forecast_out[i] = sensor_data[i+lags]

train_fc_in  = torch.tensor(forecast_in[train_indices],  dtype=torch.float32).to(device)
train_fc_out = torch.tensor(forecast_out[train_indices], dtype=torch.float32).to(device)
valid_fc_in  = torch.tensor(forecast_in[valid_indices],  dtype=torch.float32).to(device)
valid_fc_out = torch.tensor(forecast_out[valid_indices], dtype=torch.float32).to(device)

train_fc_dataset = TimeSeriesDataset(train_fc_in,  train_fc_out)
valid_fc_dataset = TimeSeriesDataset(valid_fc_in,  valid_fc_out)

print('Training SHRED forecaster...')
shred_forecaster = SHRED(num_sensors, num_sensors, hidden_size=64, hidden_layers=2, l1=100, l2=150, dropout=0.1).to(device)
fit(shred_forecaster, train_fc_dataset, valid_fc_dataset, batch_size=20, num_epochs=200, lr=1e-3, verbose=True, patience=5)

print('Training UQ forecaster...')
forecaster = UQ_Forecaster(input_size=num_sensors, hidden_size=64, hidden_layers=2, noise_dim=50).to(device)
fit_uq(forecaster, train_fc_dataset, valid_fc_dataset, batch_size=20, num_epochs=1000, lr=1e-3, verbose=True, patience=5)

print('E5 training complete')

In [ ]:
# Generate forecasts
horizon  = 50
initial  = test_data_in[0:1]

shred_forecaster.eval()
with torch.no_grad():
    history    = initial.clone()
    shred_traj = []
    for t in range(horizon):
        next_val = shred_forecaster(history)
        shred_traj.append(next_val.cpu().numpy().squeeze())
        history = torch.cat([history[:, 1:, :], next_val.unsqueeze(1)], dim=1)
shred_traj = np.array(shred_traj)

uq_traj    = forecaster.sample_trajectory(initial, horizon=horizon, n_samples=50).squeeze(1).cpu().numpy()
mean_traj  = uq_traj.mean(axis=0)
median_traj= np.median(uq_traj, axis=0)
std_traj   = uq_traj.std(axis=0)
lower_traj = np.percentile(uq_traj, 2.5,  axis=0)
upper_traj = np.percentile(uq_traj, 97.5, axis=0)

start_idx  = test_indices[0] + lags
gt_future  = sensor_data[start_idx:start_idx+horizon]

shred_fc_error    = np.linalg.norm(shred_traj - gt_future)  / np.linalg.norm(gt_future)
uq_fc_error       = np.linalg.norm(mean_traj  - gt_future)  / np.linalg.norm(gt_future)
uq_median_fc_error= np.linalg.norm(median_traj- gt_future)  / np.linalg.norm(gt_future)

print(f'E5 Forecast: SHRED={shred_fc_error:.4f}, UQ Mean={uq_fc_error:.4f}, UQ Median={uq_median_fc_error:.4f}')

In [ ]:
# E5 Visualization: Forecast Comparison
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
t   = np.arange(horizon)
si  = 0  # sensor index

axes[0].plot(t, gt_future[:, si], 'k--', linewidth=1.5, label='Ground Truth')
axes[0].plot(t, shred_traj[:, si], 'r-',  linewidth=1.2, label='SHRED Forecast')
axes[0].set_ylabel('Sensor Value')
axes[0].legend()
axes[0].set_title('SHRED Forecast (Deterministic)')
axes[0].grid(alpha=0.3)

axes[1].fill_between(t, lower_traj[:, si], upper_traj[:, si], alpha=0.3, color='blue', label='95% CI')
axes[1].plot(t, gt_future[:, si], 'k--', linewidth=1.5, label='Ground Truth')
axes[1].plot(t, mean_traj[:, si],  'b-',  linewidth=1.2, label='UQ Mean')
axes[1].plot(t, median_traj[:, si],'g--', linewidth=1.2, label='UQ Median')
axes[1].set_ylabel('Sensor Value')
axes[1].legend()
axes[1].set_title('UQ-Forecaster (With Uncertainty)')
axes[1].grid(alpha=0.3)

axes[2].plot(t, gt_future[:, si], 'k-',  linewidth=1.5, label='Ground Truth')
axes[2].plot(t, shred_traj[:, si],'r-',  linewidth=1.2, label='SHRED')
axes[2].plot(t, mean_traj[:, si], 'b-',  linewidth=1.2, label='UQ Mean')
axes[2].plot(t, median_traj[:, si],'g--',linewidth=1.2, label='UQ Median')
axes[2].fill_between(t, lower_traj[:, si], upper_traj[:, si], alpha=0.15, color='blue')
axes[2].set_xlabel('Forecast Horizon')
axes[2].set_ylabel('Sensor Value')
axes[2].legend()
axes[2].set_title('E5: Forecast Overlay Comparison')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/E5_forecast_comparison.png', dpi=150)
plt.show()

In [ ]:
# E5: Uncertainty growth over horizon
avg_std_horizon   = std_traj.mean(axis=1)
shred_err_horizon = np.abs(shred_traj - gt_future).mean(axis=1)
uq_err_horizon    = np.abs(mean_traj  - gt_future).mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(horizon), avg_std_horizon,   'b--', marker='o', markersize=3, label='UQ Uncertainty (σ)')
ax.plot(np.arange(horizon), shred_err_horizon, 'r-',  marker='s', markersize=3, alpha=0.7, label='SHRED Error')
ax.plot(np.arange(horizon), uq_err_horizon,    'b-',  marker='s', markersize=3, alpha=0.7, label='UQ Mean Error')
ax.set_xlabel('Forecast Horizon')
ax.set_ylabel('Value')
ax.set_title('E5: Uncertainty & Error Growth Over Horizon (Solar)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{results_dir}/E5_uncertainty_growth.png', dpi=150)
plt.show()

# E6: Ablation

In [ ]:
n_samples_list = [10, 25, 40, 50]
ablation_results = []

print('E6: Ablation — sampling size')
for n_samp in n_samples_list:
    samp     = uq_shred.sample(test_dataset.X, n_samples=n_samp)
    mean_pred= samp.mean(dim=0)
    rel_err  = (torch.linalg.norm(mean_pred - test_dataset.Y) / torch.linalg.norm(test_dataset.Y)).item()
    crps_val = uq.crps(samp, test_dataset.Y)
    sharp_val= uq.sharpness(samp, conf=0.95)
    cal_val  = uq.calibration_scores(samp, test_dataset.Y, levels=[0.95])[0.95]

    ablation_results.append({
        'n_samples': n_samp, 'rel_error': rel_err,
        'crps': crps_val, 'sharpness': sharp_val, 'coverage_95': cal_val
    })
    print(f'  n={n_samp}: Error={rel_err:.4f}, CRPS={crps_val:.4f}, Cov={cal_val*100:.1f}%')

n_samp_arr = np.array([r['n_samples']   for r in ablation_results])
rel_err_arr= np.array([r['rel_error']   for r in ablation_results])
crps_arr   = np.array([r['crps']        for r in ablation_results])
sharp_arr  = np.array([r['sharpness']   for r in ablation_results])
cov_arr    = np.array([r['coverage_95'] for r in ablation_results]) * 100

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(n_samp_arr, rel_err_arr, 'o-', linewidth=2, markersize=8)
axes[0, 0].axhline(shred_error, color='r', linestyle='--', label='SHRED baseline')
axes[0, 0].set_xlabel('Number of Samples')
axes[0, 0].set_ylabel('Relative Error')
axes[0, 0].set_title('Accuracy vs Sampling Size')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(n_samp_arr, crps_arr, 'o-', linewidth=2, markersize=8, color='green')
axes[0, 1].set_xlabel('Number of Samples')
axes[0, 1].set_ylabel('CRPS')
axes[0, 1].set_title('CRPS vs Sampling Size')
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(n_samp_arr, sharp_arr, 'o-', linewidth=2, markersize=8, color='purple')
axes[1, 0].set_xlabel('Number of Samples')
axes[1, 0].set_ylabel('Sharpness (95% CI width)')
axes[1, 0].set_title('Sharpness vs Sampling Size')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(n_samp_arr, cov_arr, 'o-', linewidth=2, markersize=8, color='orange')
axes[1, 1].axhline(95, color='k', linestyle='--', label='Nominal 95%')
axes[1, 1].set_xlabel('Number of Samples')
axes[1, 1].set_ylabel('Coverage (%)')
axes[1, 1].set_title('Calibration vs Sampling Size')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle('E6: Ablation Study — Solar Data', fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(f'{results_dir}/E6_ablation.png', dpi=150)
plt.show()

# Summary

In [ ]:
summary = f"""{'='*80}
UQ-SHRED RESULTS: {dataset_name}
{'='*80}
Timestamp: {timestamp}

E1 RECONSTRUCTION:
  SHRED Error:      {shred_error:.4f}
  UQ Mean Error:    {uq_mean_error:.4f}
  UQ Median Error:  {uq_median_error:.4f}
  CRPS:             {crps_score:.4f}
  Sharpness:        {sharp:.4f}
  Coverage (95%):   {cal_scores[0.95]*100:.1f}%
  Correlation:      {corr:.3f}

E5 FORECASTING:
  SHRED:            {shred_fc_error:.4f}
  UQ Mean:          {uq_fc_error:.4f}
  UQ Median:        {uq_median_fc_error:.4f}
{'='*80}
"""
print(summary)

with open(f'{results_dir}/metrics.txt', 'w') as f:
    f.write(summary)
with open(f'{results_dir}/config.json', 'w') as f:
    json.dump(config, f, indent=2)
with open(f'{results_dir}/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

print(f'\n✓ All results saved to: {results_dir}/')